# Passing Behavior with Lambdas

In this lesson, you will learn to supply behavior with lambdas, pass it to methods, and explain captured values and method references.

A method can receive more than the text it should process. It can also receive the operation to perform on that text. You will develop this idea through text rules, then write a helper that serves several rules.

Use the [module glossary](terms.md) to revisit terms after their explanations in the lesson.


## Learning Goals

By the end of this lesson, you will be able to:

- Implement a helper that accepts and applies a typed functional value.
- Compare creating a lambda, invoking its operation, and supplying a compatible method reference.
- Explain which enclosing local values a returned lambda may use.


## Why This Matters

Software often repeats a processing step while changing one choice within it. A reporting tool may clean text for one caller and add a label for another. If each choice needs its own copy of the processing method, a shared improvement must be repeated in several places. Those copies can drift apart.

Passing behavior lets one method keep its common job while callers choose the operation. The method depends on the operation's input and result types. It does not need a separate branch for every rule a caller might supply.

Earlier interfaces described operations that different objects could provide. Here, you will use an interface with one required operation to express a small piece of behavior. This also prepares you to apply operations to collection elements in later lessons.


## Check Your Starting Point

Before reading further, explain what an interface promises and how a method can accept an interface-typed parameter. Distinguish assigning a reference from invoking a method through it. Finally, recall what type arguments contribute to a generic type.


In [ ]:
Your response:




<details>
<summary>Show answer</summary>

An interface describes operations an implementation must supply. A parameter declared with that interface type can receive a compatible object reference. The method can use the interface's operations without depending on one particular implementing class.

Assigning a reference stores a value that identifies an object. It does not automatically invoke an operation on that object. A later method call requests an operation and may produce a result.

Type arguments fill the type roles declared by a generic type. Those roles depend on the type's contract. For example, a map's two arguments describe key and value types. The two arguments of the interface taught next describe different roles: an operation's input and result.

</details>


## Video Demonstration

The demonstration follows a visitor desk that needs both cleaned names and guest labels. Watch how the same helper receives a different rule for each call.

<video controls preload="metadata" width="960" style="max-width:100%;height:auto;">
  <source src="media/01_passing_behavior_with_lambdas/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/01_passing_behavior_with_lambdas/captions.vtt" srclang="en" label="English">
</video>

[Read the video transcript](media/01_passing_behavior_with_lambdas/transcript.md).


## Concept

### Describe the operation before choosing its behavior

Consider a text-processing method whose caller chooses how to format one String. Both a greeting rule and a cleaning rule accept text and return text. The particular formatting differs, but the method needs one common description of how to request the operation.

A **functional interface** describes one required abstract operation. An abstract operation specifies what an implementation must provide. A lambda can supply that one operation. This restriction matters: an interface with two unrelated required abstract operations cannot be implemented by one lambda.

Java already provides **`Function`**, a functional interface for an operation that accepts one input and returns one result. Import it from `java.util.function`. The spelling `Function<String, String>` supplies two **type arguments**: the first describes the input type, and the second describes the result type. They happen to be the same type here, but they have different jobs.

The operation is named **`apply`**. For this type of Function, `apply` receives a String and returns a String. `Function` and `apply` are library names, not Java keywords. Once that contract is clear, the caller can choose the behavior that fulfills it.


### Create a rule, then invoke it

Our first rule adds an exclamation mark to supplied text. A **lambda expression** supplies behavior for a compatible functional interface. Read this declaration as a small operation stored for later use:

```text
Function<String, String> emphasize = text -> text + "!";
```

The **target type**, `Function<String, String>`, tells Java the expected input and result types. Java uses that context to determine that the parameter `text` is a String. The **lambda parameter** names the input for one application of the rule; callers do not have to name their own variables `text`.

The arrow `->` separates the parameter from the **lambda body**, the expression `text + "!"`. The body constructs the result. In this single-expression form, its value is returned without writing a separate **`return`** statement. The arrow is syntax, not a Java keyword.

The assignment creates a functional value that can perform this operation. It does not run the body with every possible input, and it prints nothing. A later `emphasize.apply("Ready")` supplies one input and invokes the operation. The surrounding `println` then displays the returned String.

The complete program below applies the same rule twice. Each application has its own input and result.


In [ ]:
import java.util.function.Function;
Function<String, String> emphasize = text -> text + "!";
System.out.println(emphasize.apply("Ready"));
System.out.println(emphasize.apply("Go"));

The output is:

```text
Ready!
Go!
```

The first `apply` binds its parameter `text` to `"Ready"`. Concatenation produces `"Ready!"`, which the first `println` displays. The second application supplies `"Go"` and produces `"Go!"`. The rule is available for both calls; creating it and invoking it are separate actions. The original Strings are immutable, so the rule constructs results rather than editing their characters.


<details class="animation-panel" open>
<summary>Create a rule, then invoke it — show or hide animation</summary>
<p><img src="media/01_passing_behavior_with_lambdas/create-rule-then-invoke.gif" alt="Declare emphasize as compatible Function&lt;String,String&gt;; no text operation or print has occurred. Supply Ready to apply; body returns Ready!, then println displays it. Supply Go to a later apply; body returns Go!, then println displays it. Keep the rule available; display both separate input/result pairs with unchanged original Strings." width="960" style="max-width:100%;height:auto;"></p>
</details>

The assignment makes a rule available. Each later apply call supplies one input and returns one result. The same rule serves both calls; the original text inputs remain unchanged. This silent loop lasts 11 seconds. Hiding it removes the visible motion. [View the final state as a still image](media/01_passing_behavior_with_lambdas/create-rule-then-invoke_still.png).


### Let the caller supply the operation

Now imagine a greeting helper used by two callers. One wants a welcome message; another wants a shorter greeting. The common job is to apply the caller's chosen text rule and return its result.

**Behavior as an argument** means passing a functional-interface value to a method so the method can invoke its operation. The next complete example defines `TextRules.render` with two parameters:

```text
static String render(String input, Function<String, String> rule)
```

The first parameter receives the text to process. The second receives the functional value that describes the operation. The return type before `render` is String because the helper returns the processed text.

Inside the method, `return rule.apply(input);` connects those roles. First, `apply` invokes the supplied operation with `input`. Then `return` sends that operation's result back to the helper's caller. Returning text is separate from printing it.

The first caller stores a welcome rule in a variable before passing it. The second supplies a lambda directly as the second argument. The parameter's Function type provides the target type for that direct lambda too. Java still copies argument values into parameters: the helper receives a reference to the supplied functional value.


In [ ]:
import java.util.function.Function;
class TextRules {
    static String render(String input, Function<String, String> rule) {
        return rule.apply(input);
    }
}
Function<String, String> welcome = name -> "Welcome, " + name;
System.out.println(TextRules.render("Ari", welcome));
System.out.println(TextRules.render("Bo", name -> "Hello, " + name));

The two calls print:

```text
Welcome, Ari
Hello, Bo
```

For the Ari call, `input` receives `"Ari"` and `rule` receives the stored welcome rule. Applying that rule returns `"Welcome, Ari"`. For the Bo call, the directly supplied lambda produces `"Hello, Bo"`.

Both calls enter the same helper body. The helper knows it can apply one String-to-String operation; the caller determines which greeting that operation constructs. This is the same interface-based separation you used earlier, focused on one small operation.


### Keep an enclosing value available for later use

Some text rules need a choice made before their eventual input arrives. A front desk may choose a prefix now and apply it to names later. Returning a functional value lets a helper prepare that rule without already knowing the name.

In the next program, `PrefixRules.withPrefix` receives a String parameter named `prefix`. Its return type is `Function<String, String>`. The method returns behavior, rather than returning a completed greeting.

```text
return text -> prefix + text;
```

Here, `text` is the lambda's own parameter. The name `prefix` refers to the enclosing helper's parameter. A **captured local value** is an enclosing local variable or parameter value used by a lambda. Each call to `withPrefix` provides a separate prefix value for its returned rule.

The complete example chooses Guest and Staff prefixes first. Later applications supply the names Maya and Luis. This separates choosing a reusable label from applying it to a particular name.


In [ ]:
import java.util.function.Function;
class PrefixRules {
    static Function<String, String> withPrefix(String prefix) {
        return text -> prefix + text;
    }
}
Function<String, String> guest = PrefixRules.withPrefix("Guest: ");
Function<String, String> staff = PrefixRules.withPrefix("Staff: ");
System.out.println(guest.apply("Maya"));
System.out.println(staff.apply("Luis"));

The program prints:

```text
Guest: Maya
Staff: Luis
```

The first helper call creates a rule associated with `"Guest: "`. The second creates a rule associated with `"Staff: "`. The second call does not replace the first rule's prefix. When the rules are applied later, each combines its own prefix with the newly supplied name.

The calls to `withPrefix` have already returned by then. Their returned functional values remain usable. Java permits this use of an enclosing local value under the assignment rule explained next.


### Check whether a local value may be captured

A captured local variable or parameter must be **final** or **effectively final**. These terms describe its assignment history; they do not mean that the surrounding method must keep running.

The Java keyword **`final`** prevents reassignment after a variable receives its initial value. In the next example, `final String label = ...` initializes a local variable inside `withLabel`. The method can use that value in its returned lambda because the variable cannot be reassigned.

A local variable or parameter is **effectively final** when its value is not reassigned, even without the keyword. The earlier `prefix` parameter qualifies: each method call supplies its initial value, and the method never assigns another value to that parameter.

An assignment to the same parameter changes this conclusion. This replacement method is intentionally invalid and is for reading only:

```text
static Function<String, String> withPrefix(String prefix) {
    prefix = prefix.trim();
    return text -> prefix + text;
}
```

The assignment to `prefix` makes that method parameter ineligible for capture. One repair is to store the cleaned text in a different local variable and never reassign that variable. Another is to clean the argument before calling the original helper. Do not place this invalid method in an ordinary runnable cell.

The next complete, valid program demonstrates an explicit final local. Its initializer trims the helper's input and appends a colon and a space immediately. The returned lambda later joins that stored label with an item name. Those are two different times of computation.


In [ ]:
import java.util.function.Function;
class ShelfRules {
    static Function<String, String> withLabel(String text) {
        final String label = text.trim() + ": ";
        return item -> label + item;
    }
}
Function<String, String> shelf = ShelfRules.withLabel("  Shelf  ");
System.out.println(shelf.apply("atlas"));
System.out.println(shelf.apply("map"));


The output is:

```text
Shelf: atlas
Shelf: map
```

Calling `withLabel` with padded Shelf text computes `"Shelf: "` and assigns it once to the local `label`. Returning the lambda makes a rule available for later use. The two `apply` calls then supply atlas and map separately; both results use the same stored label.

If `label = "Other: ";` were inserted after its final initialization and before the return, Java would reject that reassignment. This is an illustrative invalid line, not another runnable example.

A final reference does not generally make the object it identifies immutable. The examples here use Strings, whose characters cannot be changed in place. Keep that object property separate from a variable's reassignment rule.

These capture rules concern locals and parameters inside an ordinary method. IJava handles top-level notebook variables differently. Use these complete named-method examples when investigating the rule; a top-level experiment does not establish what is allowed inside the method.


### Refer to an operation that already exists

We have written small expressions to describe new rules. Sometimes the desired operation is already a method. For example, String's `trim` operation removes the ordinary surrounding spaces used in these examples.

A **method reference** supplies compatible behavior by referring to an existing method. In the target type `Function<String, String>`, the expression `String::trim` supplies the same text operation as `text -> text.trim()`.

The double colon names the operation to use later. It does not immediately call `trim`. In this particular form, the input passed to `apply` becomes the **receiver**: the String on which `trim` runs. Thus, a later application with padded lab text invokes `trim` on that text and returns its cleaned contents.

The complete example below keeps rule creation and application on separate lines. We are studying this compatible String-to-String form, rather than every form a method reference can take.


In [ ]:
import java.util.function.Function;
Function<String, String> clean = String::trim;
System.out.println(clean.apply("  lab  "));

The program prints `lab`. The assignment stores a cleaning rule. Only the later `apply` supplies `"  lab  "` as the receiver of `trim`; `println` displays the returned contents.

The original String is unchanged. Also, `trim` does not promise to create a distinct String object on every call: an already-clean String may be returned unchanged. Our concern here is the returned text and when the operation runs.

A method reference is useful when an existing compatible method states the whole rule clearly. A lambda is useful when the rule needs an expression, such as adding a chosen label.


<details class="animation-panel" open>
<summary>Use the later input as the method receiver — show or hide animation</summary>
<p><img src="media/01_passing_behavior_with_lambdas/method-reference-receiver.gif" alt="String::trim supplies compatible behavior to clean without trimming an input. The later clean.apply call supplies the exact padded lab String. That input is the trim receiver; trim produces lab. println displays lab; preserve the original padded input as labeled comparison context." width="960" style="max-width:100%;height:auto;"></p>
</details>

Creating String::trim supplies behavior. The later input becomes the object whose trim method runs. The apply and print phases belong to one call; the diagram does not show a second invocation or promise a newly allocated object. This silent loop lasts 11 seconds. Hiding it removes the visible motion. [View the final state as a still image](media/01_passing_behavior_with_lambdas/method-reference-receiver_still.png).


## Worked Example

### Give a visitor desk one rendering helper

A visitor desk needs cleaned names for some displays and guest labels for others. The names are Strings. Spaces around a name are characters to remove for the cleaning rule; `"Guest: "` is a chosen prefix with a trailing space. The helper must apply exactly the supplied rule once and return its String result.

**Describe the common operation.** `LabelPrinter.render` receives `text` and `rule`. Its body, `return rule.apply(text);`, performs the operation and returns the result. It does not choose a prefix or decide to trim. This keeps the caller's choice outside the helper.

**Prepare the choices.** `text -> text.trim()` supplies a cleaning rule named `clean`. `text -> "Guest: " + text` supplies a labeling rule named `tag`. Assigning these functional values does not yet process Maya or Luis.

**Supply both arguments.** In `LabelPrinter.render("  Maya  ", clean)`, the first argument supplies the name and the second supplies the cleaning behavior. `render` invokes that behavior and returns its result to `println`. The Luis call supplies the tag rule instead.

**Use an existing operation.** The final rule, `namedClean`, uses `String::trim`. Applying it to padded Nora text demonstrates that the helper accepts a compatible method reference through the same Function contract.

The complete program brings these steps together.


In [ ]:
import java.util.function.Function;
class LabelPrinter {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = text -> text.trim();
Function<String, String> tag = text -> "Guest: " + text;
System.out.println(LabelPrinter.render("  Maya  ", clean));
System.out.println(LabelPrinter.render("Luis", tag));
Function<String, String> namedClean = String::trim;
System.out.println(LabelPrinter.render("  Nora  ", namedClean));


The visitor desk receives:

```text
Maya
Guest: Luis
Nora
```

For the first call, `text` receives padded Maya and `rule` receives `clean`. Applying the rule removes the surrounding spaces. The second call supplies Luis to `tag`, whose concatenation produces the guest label. The final call supplies padded Nora to the method-reference rule and returns the cleaned name.

Each call follows the same chain: caller arguments become helper parameters, `apply` invokes the selected operation, the helper returns the String, and `println` displays it. The variation belongs to the supplied rule, so the helper body stays the same.


<details class="animation-panel" open>
<summary>Pass a chosen rule through one helper — show or hide animation</summary>
<p><img src="media/01_passing_behavior_with_lambdas/pass-functional-value-through-helper.gif" alt="Caller has the clean rule and padded Maya input. render receives text and the same supplied functional value. rule.apply(text) obtains Maya from the trim body; render returns that String. Caller prints Maya; the Guest rule follows the same helper path with Luis and returns Guest: Luis. The later namedClean reference similarly produces Nora; label earlier calls as prior context, not repeated execution." width="960" style="max-width:100%;height:auto;"></p>
</details>

Each call binds both text and rule. The helper invokes the selected rule once and returns its result. Printing happens afterward in the caller. Earlier lines remain only as output history. This silent loop lasts 21 seconds. Hiding it removes the visible motion. [View the final state as a still image](media/01_passing_behavior_with_lambdas/pass-functional-value-through-helper_still.png).


## Guided Practice

Use the taught rules to trace new inputs, complete a helper, change behavior, and repair a capture error. Keep predictions separate from observations so a different result can help you find the step to revisit.


### Trace the rule selected by each caller

The complete program below uses Iris, Owen and Bea. Before running it, predict all three printed lines. For each call, identify the input String, the supplied rule and its returned result. Explain why assigning the lambda or method reference does not itself print a name or invoke the operation.


In [ ]:
Your response:

Three predicted lines:
Input, rule and result for each call:
Why assignment does not invoke or print:


In [ ]:
import java.util.function.Function;
class LabelPrinter {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = text -> text.trim();
Function<String, String> tag = text -> "Guest: " + text;
System.out.println(LabelPrinter.render("  Iris  ", clean));
System.out.println(LabelPrinter.render("Owen", tag));
Function<String, String> namedClean = String::trim;
System.out.println(LabelPrinter.render("  Bea  ", namedClean));


Record all three actual lines. Compare them with your predictions without erasing your earlier work; explain any correction.

Trace the second `render` call in detail: which arguments become `text` and `rule`, what input reaches `apply`, what result it returns, and how that result reaches `println`. Then explain why the first and third calls both trim despite their different syntax. Name Function's required abstract operation.


In [ ]:
Your response:

Actual lines and corrections:
Second-call arguments and parameters:
Input to apply, returned String and printed result:
Why both forms trim:
Function abstract operation:


<details>
<summary>Show answer</summary>

The first call passes the padded Iris String and the clean value to render. render calls `rule.apply(text)`, so the clean lambda returns Iris without surrounding spaces. The second call applies the tag lambda to Owen and returns Guest: Owen. The last call applies the `String::trim` method reference to the padded Bea String and returns Bea. These are the three printed lines, in order. Assigning a lambda or method reference supplies behavior; the render calls invoke it through apply. In `Function<String, String>`, the first String is the input type and the second is the result type. In the second call, the helper receives Owen as text and the tag functional value as rule. Calling apply invokes the supplied operation and returns its String to render, which returns it to the print statement. The clean lambda and method reference provide the same trim operation in this context; neither declaration trims every possible String in advance. Function has one abstract operation, apply.

```java
import java.util.function.Function;
class LabelPrinter {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = text -> text.trim();
Function<String, String> tag = text -> "Guest: " + text;
System.out.println(LabelPrinter.render("  Iris  ", clean));
System.out.println(LabelPrinter.render("Owen", tag));
Function<String, String> namedClean = String::trim;
System.out.println(LabelPrinter.render("  Bea  ", namedClean));
```

Expected output:

```text
Iris
Guest: Owen
Bea
```

Common error: Treating assignment of a rule as a call that processes text. Assuming every supplied rule trims the input. Reading the second type argument as a second input.

</details>


### Keep a prefix available to a returned rule

The next helper creates separate rules for desk and room labels. Predict all three printed lines before running its complete program. Mark the two `withPrefix` calls and the two later `apply` calls. For each returned rule, identify its enclosing parameter value and the input supplied later.

Explain why `prefix` is effectively final in this method. State whether `withPrefix` must still be running when its returned rule is applied.


In [ ]:
Your response:

Predicted lines:
Each captured prefix and later input:
Helper calls versus apply calls:
Capture eligibility and use after return:


In [ ]:
import java.util.function.Function;
class PrefixMaker {
    static Function<String, String> withPrefix(String prefix) {
        return text -> prefix + text;
    }
}
Function<String, String> first = PrefixMaker.withPrefix("Desk: ");
Function<String, String> second = PrefixMaker.withPrefix("Room: ");
System.out.println("Rules ready");
System.out.println(first.apply("East"));
System.out.println(second.apply("West"));


Record the three actual lines and compare them with your prediction. Explain why the two rules keep separate prefixes, when each lambda body runs, and why using the method parameter is allowed here.


In [ ]:
Your response:




<details>
<summary>Show answer</summary>

The two withPrefix calls create separate functional values using the parameter values Desk: and Room:, each followed by a space. Rules ready is printed before either apply call. first.apply("East") returns Desk: East; second.apply("West") returns Room: West. Each returned rule remains usable after the helper returns. The method parameter prefix is effectively final because the method never reassigns it. This is a local parameter inside a named method, so it is the correct place to examine the local capture rule. The later text input belongs to each apply call; it is separate from the earlier prefix value.

```java
import java.util.function.Function;
class PrefixMaker {
    static Function<String, String> withPrefix(String prefix) {
        return text -> prefix + text;
    }
}
Function<String, String> first = PrefixMaker.withPrefix("Desk: ");
Function<String, String> second = PrefixMaker.withPrefix("Room: ");
System.out.println("Rules ready");
System.out.println(first.apply("East"));
System.out.println(second.apply("West"));
```

Expected output:

```text
Rules ready
Desk: East
Room: West
```

Common error: Thinking the later text input is supplied when withPrefix is called. Assuming the second helper call replaces the prefix used by the first returned rule. Using a top-level notebook variable to draw conclusions about a method-local capture.

<details class="animation-panel" open>
<summary>Keep each returned rule’s own prefix — show or hide animation</summary>
<p><img src="media/01_passing_behavior_with_lambdas/per-call-captured-local.gif" alt="withPrefix receives Desk: and returns first, which uses that parameter value. A separate withPrefix call receives Room: and returns second with its own captured value. Both helpers have returned; Rules ready prints before either apply call. first.apply(East) returns Desk: East; second.apply(West) returns Room: West." width="960" style="max-width:100%;height:auto;"></p>
</details>

Each withPrefix call supplies a separate prefix for its returned rule. Both helper calls return before Rules ready is printed. The later East and West inputs reach their own apply calls; creating the second rule does not replace the first rule’s prefix. This silent loop lasts 13.53 seconds. Hiding it removes the visible motion. [View the final state as a still image](media/01_passing_behavior_with_lambdas/per-call-captured-local_still.png).

</details>


### Complete a method that receives a rule

Replace `FUNCTION_TYPE`, `INVOKE` and `ARROW` in the displayed program. Use `Function<String, String>` for the helper's rule parameter, the interface operation for its call, and the lambda arrow between parameter and body. Before coding, record your three substitutions and expected output. Explain why the helper receives both a String and a functional value, and why the rule returns a String.

This incomplete sample is for repair. Copy the complete repaired program into the Java work cell after the prose response and run it. It should print `Note: Map`.

```java
import java.util.function.Function;
class NotePrinter {
    static String render(String text, FUNCTION_TYPE rule) {
        return rule.INVOKE(text);
    }
}
Function<String, String> notice = text ARROW "Note: " + text;
System.out.println(NotePrinter.render("Map", notice));
```


In [ ]:
Your response:




Record the actual output from your completed program. Explain how the String argument, supplied rule, `apply` result and helper return work together. If you changed a substitution after running, explain that correction.


In [ ]:
Your response:




<details>
<summary>Show answer</summary>

Use `Function<String, String>` for FUNCTION_TYPE, apply for INVOKE and -> for ARROW. The helper accepts a String input and a compatible functional value. Its apply call runs the notice operation with Map, producing Note: Map. render returns that String; the caller prints it. The full program includes the import, helper, rule and caller, so it does not depend on an earlier cell.

```java
import java.util.function.Function;
class NotePrinter {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> notice = text -> "Note: " + text;
System.out.println(NotePrinter.render("Map", notice));
```

Expected output:

```text
Note: Map
```

Common error: Omitting the Function import in the completed program. Calling the functional value with ordinary method-call syntax instead of apply. Leaving a placeholder in executable code.

</details>


### Change the supplied labeling rule

The starter is the complete Iris, Owen and Bea prediction program. Change only the prefix inside `tag` from `"Guest: "` to `"Visitor: "`. Keep its trailing space.

Before editing, predict all three lines. Identify the line affected by the rule change and explain why the other two retain their behavior. Then edit and run the complete starter.


In [ ]:
Your response:




In [ ]:
import java.util.function.Function;
class LabelPrinter {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = text -> text.trim();
Function<String, String> tag = text -> "Guest: " + text;
System.out.println(LabelPrinter.render("  Iris  ", clean));
System.out.println(LabelPrinter.render("Owen", tag));
Function<String, String> namedClean = String::trim;
System.out.println(LabelPrinter.render("  Bea  ", namedClean));


Record all three actual lines after changing the prefix. Explain how the changed lambda affects its caller while the other rules and `LabelPrinter.render` remain usable.


In [ ]:
Your response:




### Change the caller's input

Keep the Visitor prefix. Next, change only the tag call's input from `"Owen"` to `"Kai"`. Predict all three lines and explain why the helper needs no change. After recording your prediction, make this one edit in the same complete program above and run it.


In [ ]:
Your response:




Record the actual Kai test output and compare it with your prediction. Distinguish changing the rule's prefix from changing the caller's text. Restore Owen while retaining your Visitor rule, then rerun and record the restored output.


In [ ]:
Your response:




<details>
<summary>Show answer</summary>

The tag value now supplies the expression "Visitor: " + text. Applied to Owen, it returns Visitor: Owen. The first and third rules still trim their own inputs, so their outputs remain Iris and Bea. The same helper continues to invoke whichever functional value it receives. Changing the second caller input to Kai changes only that rule’s returned label, giving Visitor: Kai. A caller value and the operation applied to it are separate choices.

```java
import java.util.function.Function;
class LabelPrinter {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = text -> text.trim();
Function<String, String> tag = text -> "Visitor: " + text;
System.out.println(LabelPrinter.render("  Iris  ", clean));
System.out.println(LabelPrinter.render("Owen", tag));
Function<String, String> namedClean = String::trim;
System.out.println(LabelPrinter.render("  Bea  ", namedClean));
```

Expected output:

```text
Iris
Visitor: Owen
Bea
```

Common error: Changing the helper to print one hard-coded visitor name. Changing clean or namedClean even though only tag’s prefix needs modification. Dropping the space after the colon.

**Additional test: `Modified tag called with Kai`.** The same Visitor rule now receives Kai. The other two inputs and rules are unchanged, so Iris and Bea remain the first and third lines.

```java
import java.util.function.Function;
class LabelPrinter {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = text -> text.trim();
Function<String, String> tag = text -> "Visitor: " + text;
System.out.println(LabelPrinter.render("  Iris  ", clean));
System.out.println(LabelPrinter.render("Kai", tag));
Function<String, String> namedClean = String::trim;
System.out.println(LabelPrinter.render("  Bea  ", namedClean));
```

Expected output:

```text
Iris
Visitor: Kai
Bea
```

</details>


### Repair a reassigned captured parameter

The displayed program is intentionally invalid. Read it before editing. Identify the parameter reassigned inside `withPrefix` and explain why Java rejects its use in the lambda. This is a compilation problem; do not predict ordinary runtime output from the invalid method.

Plan a repair that stores the formatted prefix in a different local String which is never reassigned. Let the lambda use that local value. Keep the helper calls and print statements unchanged. Explain why your proposed local qualifies for capture and predict the repaired program's two lines.

Investigate the rule inside this named method; top-level IJava variables are handled differently. Put only your complete repaired program in the Java work cell and run it.

```java
import java.util.function.Function;
class PrefixRepair {
    static Function<String, String> withPrefix(String prefix) {
        prefix = "[" + prefix + "] ";
        return text -> prefix + text;
    }
}
Function<String, String> first = PrefixRepair.withPrefix("Help");
Function<String, String> second = PrefixRepair.withPrefix("Lab");
System.out.println(first.apply("Mina"));
System.out.println(second.apply("Kai"));
```


In [ ]:
Your response:




Record the actual repaired output. Explain where the formatted prefix is computed, which local value each returned rule uses, and why the repair satisfies the capture rule.


In [ ]:
Your response:




### Test a different enclosing value

In your repaired program, change only the first helper argument from `"Help"` to `"Office"`. Predict both lines and explain which returned rule is affected. Then make that edit in the same complete program above and run it.


In [ ]:
Your response:




Record the Office test output. Explain why the Lab rule retains its own prefix. Restore Help, rerun the complete program, and record the restored output and what this check confirms.


In [ ]:
Your response:




<details>
<summary>Show answer</summary>

The faulty method assigns prefix a new formatted value before the lambda captures it. That reassignment prevents the method parameter from being effectively final. The repair initializes a new local String, labelPrefix, from the original parameter and never reassigns labelPrefix. The lambda therefore may use it. The first helper call returns a rule using [Help] followed by a space, and the second returns a rule using [Lab] followed by a space. Applying them produces [Help] Mina and [Lab] Kai. A captured local reference must meet the local-variable rule; moving the experiment to top-level notebook variables would test a different situation.

```java
import java.util.function.Function;
class PrefixRepair {
    static Function<String, String> withPrefix(String prefix) {
        String labelPrefix = "[" + prefix + "] ";
        return text -> labelPrefix + text;
    }
}
Function<String, String> first = PrefixRepair.withPrefix("Help");
Function<String, String> second = PrefixRepair.withPrefix("Lab");
System.out.println(first.apply("Mina"));
System.out.println(second.apply("Kai"));
```

Expected output:

```text
[Help] Mina
[Lab] Kai
```

Common error: Keeping the reassignment to prefix and only changing the lambda’s parameter name. Reassigning labelPrefix after initializing it. Replacing each rule with a fixed complete result, so later input no longer matters.

**Additional test: `Repaired helper receives Office instead of Help`.** Only the first captured prefix changes. The later Mina and Kai inputs and the second Lab prefix remain unchanged.

```java
import java.util.function.Function;
class PrefixRepair {
    static Function<String, String> withPrefix(String prefix) {
        String labelPrefix = "[" + prefix + "] ";
        return text -> labelPrefix + text;
    }
}
Function<String, String> first = PrefixRepair.withPrefix("Office");
Function<String, String> second = PrefixRepair.withPrefix("Lab");
System.out.println(first.apply("Mina"));
System.out.println(second.apply("Kai"));
```

Expected output:

```text
[Office] Mina
[Lab] Kai
```

</details>


## Independent Practice

### Supply separate cleaning and bracketing rules

A room-label report must clean supplied room text, then display the cleaned text inside square brackets. It also needs to show what directly bracketing empty text produces.

Define `RoomLabels.render(String text, Function<String, String> rule)` to invoke the supplied rule and return its result. Create separate functional values for `String::trim` and a lambda that surrounds its input with square brackets. Clean `"  lab  "`, then bracket the cleaned result. Also bracket empty text directly. The required lines are `[lab]` and `[]`.

Before coding, describe the input, rule and result for each application. Explain how the helper can serve both operations. Then write your own complete program in the empty Java cell, including its import, class, rules and caller setup, and run it.


In [ ]:
Your response:




Record both actual baseline lines. Trace the original room text into the cleaning call and its result into the bracketing call. Explain when each supplied operation runs, including the separate direct empty-text call, and why the same helper serves both rules.


In [ ]:
Your response:




### Check different and empty room text

Test `"  studio  "`, whitespace-only text `"   "`, and already-clean `"hall"` in place of `"  lab  "`. Keep the separate direct empty-text bracket call in every test.

Before running any variant, predict the intermediate cleaned String and both printed lines for each input. Label the cases so exact spaces and brackets can be compared later. Include the lab baseline that you will restore after testing.


In [ ]:
Your response:

Studio: cleaned String and two lines:
Whitespace only: cleaned String and two lines:
Hall: cleaned String and two lines:
Restored lab: cleaned String and two lines:


Edit and run your complete program for each planned input. Record the actual intermediate cleaned String and both output lines for studio, whitespace-only text and hall. Compare them with your predictions and explain any correction. Then restore and rerun the original lab case; record its output too.

Explain why whitespace-only cleaning followed by bracketing can agree with direct empty-text bracketing. Identify the two calls in the first route and the one call in the second. Keep the direct-empty check in every complete test.


In [ ]:
Your response:

Actual studio result:
Actual whitespace-only result:
Actual hall result:
Restored lab result:
Corrections and two-call versus one-call explanation:


<details>
<summary>Show answer</summary>

`RoomLabels.render` returns `rule.apply(text)`, so the supplied functional value determines the operation. The clean value uses `String::trim`; applied to the padded lab text, it returns lab. The bracket lambda then receives that cleaned String and returns [lab]. The separate call passes empty text directly to bracket and returns []. Cleaning and bracketing happen at their respective applications, in that order. The program includes its Function import, helper, both rules and all caller setup. The boundary cases separate a changed room, removal of all surrounding whitespace, and an input that needs no trimming. Each still applies the clean rule before the bracket rule for the room, then applies bracket directly to empty text. Restoring the original caller checks that the baseline still prints [lab] and [].

```java
import java.util.function.Function;
class RoomLabels {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = String::trim;
Function<String, String> bracket = text -> "[" + text + "]";
String room = RoomLabels.render("  lab  ", clean);
System.out.println(RoomLabels.render(room, bracket));
System.out.println(RoomLabels.render("", bracket));
```

Expected output:

```text
[lab]
[]
```

Common error: Bracketing the padded room before cleaning, leaving spaces inside the brackets. Hard-coding lab inside render instead of invoking its rule parameter. Trying to call String::trim as though it were an already computed String.

**Additional test: Different padded room: studio.** trim returns studio; bracket then returns [studio]. The independent direct empty-text application still returns [].

```java
import java.util.function.Function;
class RoomLabels {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = String::trim;
Function<String, String> bracket = text -> "[" + text + "]";
String room = RoomLabels.render("  studio  ", clean);
System.out.println(RoomLabels.render(room, bracket));
System.out.println(RoomLabels.render("", bracket));
```

Expected output:

```text
[studio]
[]
```

**Additional test: Whitespace-only room text.** The first line results from cleaning spaces to an empty String and then bracketing it. The second brackets an already empty String directly. Both produce [], but the first uses two applications and the second uses one.

```java
import java.util.function.Function;
class RoomLabels {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = String::trim;
Function<String, String> bracket = text -> "[" + text + "]";
String room = RoomLabels.render("   ", clean);
System.out.println(RoomLabels.render(room, bracket));
System.out.println(RoomLabels.render("", bracket));
```

Expected output:

```text
[]
[]
```

**Additional test: Already-clean room: hall.** trim leaves the text hall unchanged. The following bracket application returns [hall]; the separate empty application returns [].

```java
import java.util.function.Function;
class RoomLabels {
    static String render(String text, Function<String, String> rule) {
        return rule.apply(text);
    }
}
Function<String, String> clean = String::trim;
Function<String, String> bracket = text -> "[" + text + "]";
String room = RoomLabels.render("hall", clean);
System.out.println(RoomLabels.render(room, bracket));
System.out.println(RoomLabels.render("", bracket));
```

Expected output:

```text
[hall]
[]
```

</details>


## Summary

A functional interface describes one required operation. A lambda supplies behavior compatible with its target type, while a method reference can supply an existing compatible operation. Neither form invokes the operation merely by being assigned.

Passing a functional value lets a helper use a caller's chosen rule. For `Function<String, String>`, `apply` accepts one String and returns one String. A returned lambda can use an enclosing local variable or parameter that is final or effectively final.

Close the answers and, without looking back, distinguish creating a rule, passing it, and invoking it. Use one short example of each action in your explanation.


In [ ]:
Your response:




<details>
<summary>Show answer</summary>

Creating a rule supplies a compatible functional value, as in assigning `text -> text.trim()` to a Function variable. No particular name is processed by that assignment.

Passing the rule supplies that value as an argument, such as the second argument of `render`. The helper receives a reference to the selected behavior.

Invoking the rule calls `apply` with an input. That invocation runs the operation and returns a result. Returning that result from a helper and printing it are further actions. Confusing assignment with invocation hides the moment when the actual input is processed.

</details>


## Reflection

Choose a text-processing need from your field. Describe two rules that could share the same input and result types. Explain what the receiving method would know and which detail each supplied rule would decide.


In [ ]:
Your response:




You can now describe an operation separately from the code that requests it. The next lesson applies small operations to collection elements through a stream pipeline. That connection adds collection processing to the behavior-passing ideas taught here.


## Supplemental Reading

- [Java functional interfaces](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/function/package-summary.html) describes standard interfaces for supplied operations.
- [Function and its apply operation](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/function/Function.html) defines the input and result type parameters.
- [Lambda bodies and captured local values](https://docs.oracle.com/javase/specs/jls/se21/html/jls-15.html#jls-15.27.2) specifies the local-variable capture rule.
